# Battery SoH / RUL Prognostics — Combined Notebook (Days 12 – 23)

Single notebook combining the work from Day 12, 13, 14, 15 (LSTM), 16 (GRU), 17 (TCN), 19 (hyper-parameter tuning), 22 (frozen pipeline) and 23 (final test evaluation).

| Part | Content | Source |
|---|---|---|
| 1 | Setup (imports, seeds, device, Drive, paths, helpers) | all |
| 2 | Dataset preparation → saves Dataset V1 | Day 13 (first half) |
| 3 | Load & verify Dataset V1 | Day 12 |
| 4 | Linear-regression baseline | Day 13 |
| 5 | Dataset findings & leakage checklist | Day 14 |
| 6 | Common train / val / test data (+ PyTorch loaders) | Days 15–17, 19, 22 |
| 7 | LSTM (PyTorch) | Day 15 |
| 8 | GRU (PyTorch) | Day 16 |
| 9 | TCN (PyTorch) | Day 17 |
| 10 | Hyper-parameter grid search: LSTM / GRU / TCN (Keras) | Day 19 |
| 11 | Final model / pipeline freeze (Keras TCN) | Day 22 |
| 12 | Final held-out test evaluation | Day 23 |

**Notes**
- Run top to bottom. If Dataset V1 already exists on Drive you can skip Part 2.
- Parts 7–9 (PyTorch) use SoH scaled to **0–1**; Parts 10–12 (Keras) use SoH on its original **0–100** scale, exactly as in the original notebooks. Error values from the two groups are therefore **not directly comparable**.
- Part 10 runs 3 × 81 experiments and takes a long time. Set `RUN_GRID_SEARCH = False` in that part to reload saved CSV results instead.

# Part 1 — Setup

In [ ]:
# ============================================================
# Imports (single place for the whole notebook)
# ============================================================
import os
import time
import json
import random
import zipfile
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# PyTorch (Days 15-17)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# TensorFlow / Keras (Days 19, 22, 23)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Let TensorFlow share the GPU with PyTorch (must run before any TF op)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError as e:
    print("GPU memory-growth setting skipped:", e)

In [ ]:
# ============================================================
# Reproducibility, device and shared configuration
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32

print("Random seed :", SEED)
print("Device      :", device)
print("Batch size  :", BATCH_SIZE)
print("TensorFlow  :", tf.__version__)

In [ ]:
# ============================================================
# Mount Google Drive and define dataset paths
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

DATASET_PATH  = "/content/drive/MyDrive/IITG dataset/cleaned_dataset"
METADATA_PATH = os.path.join(DATASET_PATH, "metadata.csv")
V1_PATH       = os.path.join(DATASET_PATH, "processed_dataset_v1")
SEQUENCE_FILE = os.path.join(V1_PATH, "sequences_v1.npz")

print("Dataset path :", DATASET_PATH)
print("V1 path      :", V1_PATH)

In [ ]:
# ============================================================
# Shared plotting helper (used by baseline, LSTM, GRU, TCN, final test)
# ============================================================
def plot_pred_vs_actual(y_true, y_pred, xlabel, ylabel, title):
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.6)

    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))

    plt.plot([min_val, max_val], [min_val, max_val],
             "r--", label="Ideal Prediction")

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_loss_curves(train_losses, val_losses, title, ylabel="Loss"):
    epochs_completed = range(1, len(train_losses) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_completed, train_losses, label="Train Loss")
    plt.plot(epochs_completed, val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Part 2 — Dataset Preparation (Day 13, first half)
Builds SoH, RUL, scaled features, chronological splits and overlapping sequences from `metadata.csv`, then saves **Dataset Version 1**.
Skip this part if `sequences_v1.npz` already exists on Drive.

## 2.1 Load metadata and select discharge cycles

In [ ]:
metadata = pd.read_csv(METADATA_PATH)

print("Dataset shape:", metadata.shape)
display(metadata.head())

In [ ]:
discharge_df = metadata[
    (metadata["type"] == "discharge") &
    (pd.to_numeric(metadata["Capacity"], errors="coerce").notna())
].copy()

discharge_df["Capacity"] = pd.to_numeric(
    discharge_df["Capacity"], errors="coerce"
)

discharge_df = discharge_df.sort_values(
    ["battery_id", "test_id"]
).reset_index(drop=True)

print("Discharge records:", len(discharge_df))
display(
    discharge_df[
        ["battery_id", "test_id", "Capacity"]
    ].head(10)
)

## 2.2 State of Health (SoH) and end-of-life (EOL)

In [ ]:
discharge_df["reference_capacity"] = (
    discharge_df.groupby("battery_id")["Capacity"]
    .transform("max")
)

discharge_df["SoH"] = (
    discharge_df["Capacity"] /
    discharge_df["reference_capacity"]
) * 100

print("SoH range:")
print(
    discharge_df["SoH"].min(),
    "to",
    discharge_df["SoH"].max()
)

display(
    discharge_df[
        ["battery_id", "test_id", "Capacity", "reference_capacity", "SoH"]
    ].head(10)
)

In [ ]:
EOL_SOH_THRESHOLD = 80.0

print("EOL SoH threshold:", EOL_SOH_THRESHOLD, "%")

eol_check = (
    discharge_df.groupby("battery_id")["SoH"]
    .min()
    .reset_index(name="minimum_SoH")
)

eol_check["reaches_EOL"] = (
    eol_check["minimum_SoH"] <= EOL_SOH_THRESHOLD
)

print("Batteries reaching EOL:",
      eol_check["reaches_EOL"].sum(),
      "out of",
      len(eol_check))

display(eol_check)

## 2.3 Remaining Useful Life (RUL)

In [ ]:
# Cycle index per battery
discharge_df["cycle_index"] = (
    discharge_df.groupby("battery_id").cumcount()
)

discharge_df["RUL"] = np.nan

for battery_id in discharge_df["battery_id"].unique():

    battery_mask = discharge_df["battery_id"] == battery_id

    battery_data = discharge_df.loc[battery_mask]

    eol_cycles = battery_data.loc[
        battery_data["SoH"] <= EOL_SOH_THRESHOLD,
        "cycle_index"
    ]

    if len(eol_cycles) > 0:

        eol_cycle = eol_cycles.iloc[0]

        rul_values = (
            eol_cycle -
            discharge_df.loc[battery_mask, "cycle_index"]
        ).clip(lower=0)

        discharge_df.loc[battery_mask, "RUL"] = rul_values

print("RUL created successfully.")

display(
    discharge_df[
        ["battery_id", "cycle_index", "SoH", "RUL"]
    ].head(15)
)

In [ ]:
print("Minimum RUL:", discharge_df["RUL"].min())
print("Maximum RUL:", discharge_df["RUL"].max())

print(
    "Negative RUL values:",
    (discharge_df["RUL"] < 0).sum()
)

## 2.4 Sequence configuration

In [ ]:
SEQUENCE_LENGTH = 10
STRIDE = 1

print("Sequence length:", SEQUENCE_LENGTH)
print("Stride:", STRIDE)

# Check available cycles for each battery

battery_cycle_counts = (
    discharge_df.groupby("battery_id")
    .size()
    .reset_index(name="num_cycles")
    .sort_values("num_cycles")
)

display(battery_cycle_counts)

print("Total batteries:", len(battery_cycle_counts))

print(
    "Batteries with at least",
    SEQUENCE_LENGTH,
    "cycles:",
    (battery_cycle_counts["num_cycles"] >= SEQUENCE_LENGTH).sum()
)

print(
    "Batteries with fewer than",
    SEQUENCE_LENGTH,
    "cycles:",
    (battery_cycle_counts["num_cycles"] < SEQUENCE_LENGTH).sum()
)

## 2.5 Feature dataframe

In [ ]:
# Create feature dataframe

feature_df = discharge_df.copy()

# Sort by battery and cycle
feature_df = feature_df.sort_values(
    ["battery_id", "cycle_index"]
).reset_index(drop=True)

# Calculate cycle progress for each battery
feature_df["cycle_progress"] = (
    feature_df.groupby("battery_id")["cycle_index"]
    .transform(
        lambda x: x / max(x.max(), 1)
    )
)

# Use ambient temperature if available
if "ambient_temperature" in feature_df.columns:

    feature_df["ambient_temperature"] = pd.to_numeric(
        feature_df["ambient_temperature"],
        errors="coerce"
    )

elif "Temperature_measured" in feature_df.columns:

    feature_df["ambient_temperature"] = pd.to_numeric(
        feature_df["Temperature_measured"],
        errors="coerce"
    )

else:

    # If temperature is not present in metadata
    feature_df["ambient_temperature"] = 0.0

# Fill only missing temperature values
feature_df["ambient_temperature"] = (
    feature_df["ambient_temperature"]
    .interpolate()
    .bfill()
    .ffill()
)

print("Feature dataframe created successfully.")

display(
    feature_df[
        [
            "battery_id",
            "test_id",
            "cycle_index",
            "cycle_progress",
            "ambient_temperature",
            "SoH",
            "RUL"
        ]
    ].head(20)
)

## 2.6 Chronological train / validation / test split (70 / 15 / 15 per battery)

In [ ]:
# Create chronological train / validation / test split

feature_df = feature_df.sort_values(
    ["battery_id", "test_id"]
).reset_index(drop=True)

feature_df["split"] = ""

for battery_id, group in feature_df.groupby("battery_id"):

    indices = group.index
    n = len(indices)

    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    feature_df.loc[
        indices[:train_end],
        "split"
    ] = "train"

    feature_df.loc[
        indices[train_end:val_end],
        "split"
    ] = "validation"

    feature_df.loc[
        indices[val_end:],
        "split"
    ] = "test"

print("Split counts:")
print(feature_df["split"].value_counts())

display(
    feature_df[
        ["battery_id", "test_id", "cycle_index", "split"]
    ].head(20)
)

# Split summary by battery

split_summary = (
    feature_df
    .groupby(["battery_id", "split"])
    .size()
    .unstack(fill_value=0)
)

display(split_summary)

print("\nOverall split counts:")
print(feature_df["split"].value_counts())

## 2.7 Feature scaling (fit on training data only)

In [ ]:
# Feature scaling

INPUT_FEATURES = [
    "cycle_progress",
    "ambient_temperature"
]

TARGET_FEATURES = [
    "SoH",
    "RUL"
]

scaler = MinMaxScaler()

# Fit scaler only on training data
train_features = feature_df.loc[
    feature_df["split"] == "train",
    INPUT_FEATURES
]

scaler.fit(train_features)

# Transform all data
feature_df[
    [
        "cycle_progress_scaled",
        "ambient_temperature_scaled"
    ]
] = scaler.transform(
    feature_df[INPUT_FEATURES]
)

print("Scaling completed.")
print("Scaler fitted only on training data.")

display(
    feature_df[
        [
            "battery_id",
            "cycle_index",
            "split",
            "cycle_progress_scaled",
            "ambient_temperature_scaled"
        ]
    ].head(10)
)

## 2.8 Create overlapping sequences

In [ ]:
# Create overlapping sequences

FINAL_FEATURES = [
    "cycle_progress_scaled",
    "ambient_temperature_scaled"
]

X_sequences = []
y_soh = []
y_rul = []

sequence_battery_ids = []
sequence_splits = []

for battery_id, battery_data in feature_df.groupby("battery_id"):

    battery_data = battery_data.sort_values(
        "cycle_index"
    ).reset_index(drop=True)

    # Skip only batteries that genuinely have
    # fewer cycles than the required sequence length
    if len(battery_data) < SEQUENCE_LENGTH:
        continue

    for start in range(
        0,
        len(battery_data) - SEQUENCE_LENGTH + 1,
        STRIDE
    ):

        end = start + SEQUENCE_LENGTH

        window = battery_data.iloc[start:end]

        # Target from final cycle
        soh_target = window["SoH"].iloc[-1]
        rul_target = window["RUL"].iloc[-1]

        # Skip sequence only if target is unavailable
        if pd.isna(soh_target) or pd.isna(rul_target):
            continue

        X_sequences.append(
            window[FINAL_FEATURES].values
        )

        y_soh.append(soh_target)
        y_rul.append(rul_target)

        sequence_battery_ids.append(
            battery_id
        )

        sequence_splits.append(
            window["split"].iloc[-1]
        )

# Convert to NumPy arrays
X_sequences = np.array(X_sequences)
y_soh = np.array(y_soh)
y_rul = np.array(y_rul)

sequence_battery_ids = np.array(
    sequence_battery_ids
)

sequence_splits = np.array(
    sequence_splits
)

print("\nFinal sequence dataset created")
print("--------------------------------")

print("Sequence length:", SEQUENCE_LENGTH)
print("Stride:", STRIDE)

print("X shape:", X_sequences.shape)
print("SoH target shape:", y_soh.shape)
print("RUL target shape:", y_rul.shape)

print(
    "Number of batteries:",
    len(np.unique(sequence_battery_ids))
)

print("\nSequence split:")
print(
    pd.Series(sequence_splits).value_counts()
)

## 2.9 Save Dataset Version 1

In [ ]:
# Save processed Dataset Version 1

OUTPUT_PATH = os.path.join(
    DATASET_PATH,
    "processed_dataset_v1"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Convert sequence metadata to arrays
sequence_battery_ids = np.array(sequence_battery_ids)
sequence_splits = np.array(sequence_splits)

# Save processed sequences and targets
np.savez_compressed(
    os.path.join(OUTPUT_PATH, "sequences_v1.npz"),
    X=X_sequences,
    SoH=y_soh,
    RUL=y_rul,
    battery_id=sequence_battery_ids,
    split=sequence_splits
)

# Save scaler
joblib.dump(
    scaler,
    os.path.join(OUTPUT_PATH, "feature_scaler_v1.pkl")
)

# Save processing metadata
processing_info = {
    "version": "Dataset_V1",
    "sequence_length": SEQUENCE_LENGTH,
    "stride": STRIDE,
    "input_features": FINAL_FEATURES,
    "targets": ["SoH", "RUL"],
    "eol_soh_threshold": EOL_SOH_THRESHOLD,
    "random_seed": SEED,
    "split_strategy": "70/15/15 chronological within each battery",
    "number_of_sequences": len(X_sequences),
    "number_of_batteries": len(set(sequence_battery_ids))
}

pd.DataFrame([processing_info]).to_json(
    os.path.join(OUTPUT_PATH, "processing_info_v1.json"),
    orient="records",
    indent=4
)

print("Dataset Version 1 saved successfully.")
print("Saved at:", OUTPUT_PATH)

## 2.10 RUL checks

In [ ]:
# RUL distribution
plt.figure(figsize=(10, 6))

# Create more bins for better distribution visibility
bins = np.linspace(
    np.min(y_rul),
    np.max(y_rul),
    51
)

plt.hist(
    y_rul,
    bins=bins,
    edgecolor="black",
    linewidth=0.7
)

plt.xlabel("RUL (cycles)")
plt.ylabel("Number of sequences")
plt.title("RUL Distribution")

# Log scale makes smaller frequency bars visible
plt.yscale("log")

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# Identify batteries with the lowest and highest RUL coverage

rul_battery_summary = (
    feature_df[feature_df["RUL"].notna()]
    .groupby("battery_id")["RUL"]
    .agg(["min", "max", "count"])
    .sort_values("min")
)

display(rul_battery_summary)

In [ ]:
#how many cycles are being rejected due to missing RUL

total_cycles = len(discharge_df)

cycles_with_rul = discharge_df["RUL"].notna().sum()
cycles_without_rul = discharge_df["RUL"].isna().sum()

print("Total cycles:", total_cycles)
print("Cycles retained for RUL:", cycles_with_rul)
print("Cycles rejected due to missing RUL:", cycles_without_rul)

print(
    "Percentage retained:",
    round(cycles_with_rul / total_cycles * 100, 2),
    "%"
)

print(
    "Percentage rejected:",
    round(cycles_without_rul / total_cycles * 100, 2),
    "%"
)

# Part 3 — Load & Verify Dataset V1 (Day 12)
Loads the saved sequence file. From here on every later part works from this loaded data, so the notebook can also be run starting at this part.

In [ ]:
# Load sequence dataset (Dataset Version 1)
data = np.load(SEQUENCE_FILE, allow_pickle=True)

X_sequences          = data["X"]
y_soh                = data["SoH"]          # original scale: 0-100
y_rul                = data["RUL"]
sequence_battery_ids = data["battery_id"]
sequence_splits      = data["split"]

print("Dataset Version 1 loaded successfully")
print("X shape  :", X_sequences.shape)
print("SoH shape:", y_soh.shape)
print("RUL shape:", y_rul.shape)

In [ ]:
# Verify the saved feature scaler
scaler_path = os.path.join(V1_PATH, "feature_scaler_v1.pkl")

loaded_scaler = joblib.load(scaler_path)

print("Scaler loaded successfully")
print("Scaling method: Min-Max Scaling")
print("Feature range:", loaded_scaler.feature_range)
print("Number of features scaled:", loaded_scaler.n_features_in_)

In [ ]:
print("===== DATASET VERSION 1 FINAL VERIFICATION =====")

print("\nSequence Configuration")
print("-----------------------")
print("Total sequences        :", len(X_sequences))
print("Sequence shape         :", X_sequences.shape)
print("Sequence length        :", X_sequences.shape[1])
print("Number of input features:", X_sequences.shape[2])
print("Unique batteries       :", len(np.unique(sequence_battery_ids)))

print("\nInput features used in Dataset V1:")
print("1. cycle_progress_scaled")
print("2. ambient_temperature_scaled")

print("\nTargets:")
print("1. SoH")
print("2. RUL")

print("\nSplit counts:")
print(pd.Series(sequence_splits).value_counts())

print("\nTarget check:")
print("SoH targets:", len(y_soh))
print("RUL targets:", len(y_rul))

print("\nDataset V1 status: VERIFIED")

# Part 4 — Linear-Regression Baseline (Day 13)

In [ ]:
X_baseline = X_sequences[:, -1, :]

print("Baseline input shape:", X_baseline.shape)
print("SoH target shape:", y_soh.shape)
print("RUL target shape:", y_rul.shape)

# Chronological split
n = len(X_baseline)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

Xb_train = X_baseline[:train_end]
Xb_val = X_baseline[train_end:val_end]
Xb_test = X_baseline[val_end:]

y_soh_train = y_soh[:train_end]
y_soh_val = y_soh[train_end:val_end]
y_soh_test = y_soh[val_end:]

y_rul_train = y_rul[:train_end]
y_rul_val = y_rul[train_end:val_end]
y_rul_test = y_rul[val_end:]

print("\nData split:")
print("Training sequences   :", len(Xb_train))
print("Validation sequences :", len(Xb_val))
print("Testing sequences    :", len(Xb_test))

In [ ]:
soh_model = LinearRegression()

# Train
soh_model.fit(
    Xb_train,
    y_soh_train
)

# Predict on test data
soh_pred = soh_model.predict(
    Xb_test
)

print("SoH baseline model trained successfully.")
print("Number of SoH predictions:", len(soh_pred))

In [ ]:
rul_model = LinearRegression()

# Train
rul_model.fit(
    Xb_train,
    y_rul_train
)

# Predict on test data
rul_pred = rul_model.predict(
    Xb_test
)

print("RUL baseline model trained successfully.")
print("Number of RUL predictions:", len(rul_pred))

In [ ]:
def calculate_metrics(y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = mean_absolute_percentage_error(
        y_true,
        y_pred
    ) * 100

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mae, rmse, mape, r2


# SoH metrics
soh_mae, soh_rmse, soh_mape, soh_r2 = calculate_metrics(
    y_soh_test,
    soh_pred
)

# RUL metrics
rul_mae, rul_rmse, rul_mape, rul_r2 = calculate_metrics(
    y_rul_test,
    rul_pred
)


# Create results table
baseline_results = pd.DataFrame({
    "Target": ["SoH", "RUL"],
    "MAE": [soh_mae, rul_mae],
    "RMSE": [soh_rmse, rul_rmse],
    "MAPE (%)": [soh_mape, rul_mape],
    "R²": [soh_r2, rul_r2]
})

display(baseline_results)

In [ ]:
plot_pred_vs_actual(
    y_soh_test, soh_pred,
    "Actual SoH (%)", "Predicted SoH (%)",
    "Baseline: Predicted vs Actual SoH"
)

plot_pred_vs_actual(
    y_rul_test, rul_pred,
    "Actual RUL (cycles)", "Predicted RUL (cycles)",
    "Baseline: Predicted vs Actual RUL"
)

# Part 5 — Dataset Findings & Leakage Checklist (Day 14)

## 1. My Dataset – Sequence Findings

The processed sequence dataset contains battery degradation sequences
prepared for modelling.

The dataset contains:

- 2287 total sequences
- Sequence length of 10
- 2 input features
- SoH target
- RUL target

The sequences are divided into training, validation and test sets.

In [ ]:
# Dataset findings (computed from the loaded Dataset V1)
my_dataset = {
    "Total sequences": len(X_sequences),
    "Sequence length": X_sequences.shape[1],
    "Input features": X_sequences.shape[2],
    "Training sequences": int(np.sum(sequence_splits == "train")),
    "Validation sequences": int(np.sum(sequence_splits == "validation")),
    "Test sequences": int(np.sum(sequence_splits == "test"))
}

my_dataset_df = pd.DataFrame(
    list(my_dataset.items()),
    columns=["Parameter", "Value"]
)

display(my_dataset_df)

In [ ]:
# Sequence split counts
split_counts = pd.Series({
    "Train": my_dataset["Training sequences"],
    "Validation": my_dataset["Validation sequences"],
    "Test": my_dataset["Test sequences"]
})

print("Sequence Split Distribution")
print("---------------------------")

display(split_counts.to_frame(name="Number of Sequences"))

In [ ]:
plt.figure(figsize=(8, 5))

split_counts.plot(
    kind="bar"
)

plt.title("Train, Validation and Test Sequence Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Sequences")

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Leakage checklist

In [ ]:
leakage_checklist = pd.DataFrame({
    "Check": [
        "Train/validation/test separation",
        "Test data used during training",
        "Validation data used for final testing",
        "Future information entering earlier sequences",
        "Overlapping trajectories across splits"
    ],

    "Status": [
        "To be monitored",
        "Should be avoided",
        "Should be avoided",
        "Potential risk",
        "Potential risk"
    ]
})

display(leakage_checklist)

# Part 6 — Common Train / Validation / Test Data
Splits come from the `split` labels stored in Dataset V1 (created in Part 2).

- `y_train / y_val / y_test` — SoH on the original **0–100** scale (Keras models, Parts 10–12)
- `y_*_norm` — SoH scaled to **0–1** (PyTorch models, Parts 7–9)

In [ ]:
# Masks from the split labels
train_mask = sequence_splits == "train"
val_mask   = sequence_splits == "validation"
test_mask  = sequence_splits == "test"

# Inputs
X_train = X_sequences[train_mask]
X_val   = X_sequences[val_mask]
X_test  = X_sequences[test_mask]

# SoH targets - original scale (0-100)
y_train = y_soh[train_mask]
y_val   = y_soh[val_mask]
y_test  = y_soh[test_mask]

# SoH targets - normalised to 0-1 (PyTorch models)
print("Original SoH scale")
print("------------------")
print("Minimum:", y_soh.min())
print("Maximum:", y_soh.max())
print("Mean   :", y_soh.mean())

y_soh_norm = y_soh / 100.0

print("\nNormalized SoH scale")
print("--------------------")
print("Minimum:", y_soh_norm.min())
print("Maximum:", y_soh_norm.max())
print("Mean   :", y_soh_norm.mean())

y_train_norm = y_soh_norm[train_mask]
y_val_norm   = y_soh_norm[val_mask]
y_test_norm  = y_soh_norm[test_mask]

seq_len   = X_train.shape[1]
input_dim = X_train.shape[2]

print("\nTRAIN      :", X_train.shape, y_train.shape)
print("VALIDATION :", X_val.shape, y_val.shape)
print("TEST       :", X_test.shape, y_test.shape)

In [ ]:
# PyTorch datasets and loaders (SoH normalised to 0-1)
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train_norm, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val_norm, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test_norm, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created successfully.")
print("Batch size:", BATCH_SIZE)

In [ ]:
# ============================================================
# Shared PyTorch helpers (training, prediction, metrics)
# ============================================================
def train_with_early_stopping(
    model, criterion, optimizer, checkpoint_path,
    num_epochs=100, patience=15
):
    """Train with gradient clipping, best-model checkpointing and early stopping."""

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    start_time = time.time()

    for epoch in range(num_epochs):

        # ---------------- TRAINING ----------------
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()

            # Prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)

        # ---------------- VALIDATION ----------------
        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:

                X_val_batch = X_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                val_outputs = model(X_val_batch)
                running_val_loss += criterion(val_outputs, y_val_batch).item()

        epoch_val_loss = running_val_loss / len(val_loader)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        # ---------------- BEST MODEL ----------------
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"Epoch [{epoch+1:03d}/{num_epochs}] "
            f"Train Loss: {epoch_train_loss:.6f} "
            f"Val Loss: {epoch_val_loss:.6f}"
        )

        # ---------------- EARLY STOPPING ----------------
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    training_time = time.time() - start_time

    print("\nTraining completed.")
    print("Training Time:", round(training_time, 2), "seconds")
    print("Best Epoch:", best_epoch)
    print("Best Validation Loss:", best_val_loss)
    print("Checkpoint:", checkpoint_path)

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "training_time": training_time
    }


def load_best_checkpoint(model, checkpoint_path):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            "Best model checkpoint was not created. "
            "Please run the training cell first."
        )
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model


def torch_predict(model, loader):
    """Return (predictions, targets, inference_time_in_seconds)."""
    model.eval()
    all_predictions, all_targets = [], []

    start_inference = time.time()

    with torch.no_grad():
        for batch_X, batch_y in loader:
            predictions = model(batch_X.to(device))
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(batch_y.numpy())

    inference_time = time.time() - start_inference

    return np.concatenate(all_predictions), np.concatenate(all_targets), inference_time


def soh_metrics(y_true, y_pred, mape_threshold=0.20, abs_r2=False):
    """MAE, RMSE, MAPE (only where SoH >= threshold) and R2.
    abs_r2=True keeps the |R2| convention used in the original LSTM/GRU notebooks."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mask = y_true >= mape_threshold
    if np.sum(mask) > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan

    r2 = r2_score(y_true, y_pred)
    if abs_r2:
        r2 = abs(r2)

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2,
            "mape_samples": int(np.sum(mask)), "mape_threshold": mape_threshold}


def print_soh_metrics(name, m, n_total):
    print(f"{name} Test Set Performance")
    print("=" * 40)
    print(f"MAE  : {m['MAE']:.6f}")
    print(f"RMSE : {m['RMSE']:.6f}")
    print(f"MAPE (SoH >= {m['mape_threshold']:.2f}): {m['MAPE']:.4f}%")
    print(f"R²   : {m['R2']:.6f}")
    print(f"\nMAPE samples used: {m['mape_samples']} / {n_total}")

# Part 7 — LSTM Model (PyTorch, Day 15)

In [ ]:
class PrognosticLSTM(nn.Module):

    def __init__(self, input_dim, hidden_dim=128, num_layers=2, output_dim=1):
        super(PrognosticLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )

        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)

        # Keep SoH between 0 and 1
        out = torch.sigmoid(out)

        return out.squeeze(-1)


lstm_model = PrognosticLSTM(
    input_dim=input_dim,
    hidden_dim=128,
    num_layers=2,
    output_dim=1
).to(device)

lstm_criterion = nn.SmoothL1Loss()

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

print(lstm_model)
print("\nLoss:", lstm_criterion)
print("Optimizer:", lstm_optimizer)

## Training the LSTM

In [ ]:
LSTM_CHECKPOINT = "/content/best_lstm_model.pth"

lstm_history = train_with_early_stopping(
    lstm_model, lstm_criterion, lstm_optimizer,
    checkpoint_path=LSTM_CHECKPOINT,
    num_epochs=100, patience=15
)

training_time_lstm = lstm_history["training_time"]

In [ ]:
plot_loss_curves(
    lstm_history["train_losses"],
    lstm_history["val_losses"],
    "LSTM Training vs Validation Loss",
    ylabel="SmoothL1 Loss"
)

## Evaluating the best LSTM

In [ ]:
load_best_checkpoint(lstm_model, LSTM_CHECKPOINT)

print("Best LSTM model loaded successfully.")
print("Best Epoch:", lstm_history["best_epoch"])
print("Best Validation Loss:", lstm_history["best_val_loss"])

y_pred_lstm, y_true_lstm, inference_time_lstm = torch_predict(lstm_model, test_loader)

# Make sure predictions are physically valid
y_pred_lstm = np.clip(y_pred_lstm, 0.0, 1.0)

num_params_lstm = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)

print(f"\nInference Time: {inference_time_lstm:.4f} seconds")
print(f"Test Samples: {len(y_true_lstm)}")
print(f"Trainable Parameters: {num_params_lstm:,}")
print("\nFirst 10 predictions:")
print(y_pred_lstm[:10])

In [ ]:
metrics_lstm = soh_metrics(y_true_lstm, y_pred_lstm, abs_r2=True)

print_soh_metrics("LSTM", metrics_lstm, len(y_true_lstm))

In [ ]:
plot_pred_vs_actual(
    y_true_lstm, y_pred_lstm,
    "Actual SoH", "Predicted SoH",
    "LSTM Test Set: Predicted vs Actual SoH"
)

# Part 8 — GRU Model (PyTorch, Day 16)

In [ ]:
class PrognosticGRU(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=2,
        output_dim=1
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):

        out, _ = self.gru(x)

        # Last time step
        out = out[:, -1, :]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        # SoH must remain between 0 and 1
        out = torch.sigmoid(out)

        return out.squeeze(-1)


gru_model = PrognosticGRU(
    input_dim=input_dim,
    hidden_dim=64,
    num_layers=2,
    output_dim=1
).to(device)

gru_criterion = nn.SmoothL1Loss()

gru_optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

print(gru_model)
print("\nLoss:", gru_criterion)
print("Optimizer:", gru_optimizer)

## Training the GRU

In [ ]:
GRU_CHECKPOINT = "/content/best_gru_model.pth"

gru_history = train_with_early_stopping(
    gru_model, gru_criterion, gru_optimizer,
    checkpoint_path=GRU_CHECKPOINT,
    num_epochs=100, patience=15
)

training_time_gru = gru_history["training_time"]

In [ ]:
plot_loss_curves(
    gru_history["train_losses"],
    gru_history["val_losses"],
    "GRU Training vs Validation Loss",
    ylabel="SmoothL1 Loss"
)

## Evaluating the best GRU

In [ ]:
load_best_checkpoint(gru_model, GRU_CHECKPOINT)

print("Best GRU model loaded.")
print("Best Epoch:", gru_history["best_epoch"])
print("Best Validation Loss:", gru_history["best_val_loss"])

y_pred_gru, y_true_gru, inference_time_gru = torch_predict(gru_model, test_loader)

print("\nPrediction completed.")
print("Actual:", y_true_gru.shape)
print("Predicted:", y_pred_gru.shape)
print(f"GRU Inference Time: {inference_time_gru:.4f} seconds")
print(f"Test Samples: {len(y_true_gru)}")

In [ ]:
metrics_gru = soh_metrics(y_true_gru, y_pred_gru, abs_r2=True)

print_soh_metrics("GRU", metrics_gru, len(y_true_gru))

In [ ]:
plot_pred_vs_actual(
    y_true_gru, y_pred_gru,
    "Actual SoH", "Predicted SoH",
    "GRU: Predicted vs Actual SoH"
)

## GRU summary

In [ ]:
total_parameters_gru = sum(p.numel() for p in gru_model.parameters())

print("GRU MODEL SUMMARY")
print("=" * 40)

print("Input Features:", input_dim)
print("Sequence Length:", seq_len)
print("Hidden Size:", 64)
print("GRU Layers:", 2)
print("Dropout:", 0.2)
print("Batch Size:", BATCH_SIZE)
print("Epochs:", 100)

print("\nBest Epoch:", gru_history["best_epoch"])
print("Best Validation Loss:", gru_history["best_val_loss"])

print("\nTest Performance")
print("----------------")
print("MAE :", metrics_gru["MAE"])
print("RMSE:", metrics_gru["RMSE"])
print("MAPE:", metrics_gru["MAPE"])
print("R²  :", metrics_gru["R2"])

print("\nTotal Parameters:", total_parameters_gru)

print("\nCheckpoint:")
print(GRU_CHECKPOINT)

# Part 9 — TCN Model (PyTorch, Day 17)

In [ ]:
class Chomp1d(nn.Module):

    def __init__(self, chomp_size):
        super().__init__()

        self.chomp_size = chomp_size

    def forward(self, x):

        if self.chomp_size > 0:
            return x[:, :, :-self.chomp_size]

        return x


class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation,
        dropout=0.2
    ):
        super().__init__()

        padding = (
            kernel_size - 1
        ) * dilation

        self.conv1 = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp1 = Chomp1d(
            padding
        )

        self.relu1 = nn.ReLU()

        self.drop1 = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv1d(
            out_ch,
            out_ch,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp2 = Chomp1d(
            padding
        )

        self.relu2 = nn.ReLU()

        self.drop2 = nn.Dropout(
            dropout
        )

        self.downsample = (
            nn.Conv1d(
                in_ch,
                out_ch,
                kernel_size=1
            )
            if in_ch != out_ch
            else None
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        out = self.conv1(x)

        out = self.chomp1(out)

        out = self.relu1(out)

        out = self.drop1(out)

        out = self.conv2(out)

        out = self.chomp2(out)

        out = self.relu2(out)

        out = self.drop2(out)

        if self.downsample is None:
            residual = x
        else:
            residual = self.downsample(x)

        return self.relu(
            out + residual
        )


class PrognosticTCN(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        output_dim=1,
        channels=(96, 96, 64),
        kernel_size=3,
        dropout=0.2
    ):
        super().__init__()

        layers = []

        in_ch = input_dim

        for i, out_ch in enumerate(channels):

            dilation = 2 ** i

            layers.append(
                TemporalBlock(
                    in_ch,
                    out_ch,
                    kernel_size,
                    dilation,
                    dropout
                )
            )

            in_ch = out_ch

        self.tcn = nn.Sequential(
            *layers
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(
            channels[-1] * seq_len,
            64
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc2 = nn.Linear(
            64,
            output_dim
        )

    def forward(self, x):

        # (batch, sequence, features)
        # → (batch, features, sequence)

        x = x.transpose(1, 2)

        x = self.tcn(x)

        x = self.flatten(x)

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x.squeeze(-1)

In [ ]:
tcn_model = PrognosticTCN(
    input_dim=input_dim,
    seq_len=seq_len,
    output_dim=1
).to(device)

tcn_criterion = nn.MSELoss()

tcn_optimizer = torch.optim.Adam(
    tcn_model.parameters(),
    lr=0.001
)

num_params_tcn = sum(
    p.numel()
    for p in tcn_model.parameters()
    if p.requires_grad
)

print(tcn_model)

print("\nTrainable parameters:", num_params_tcn)

## Training the TCN

In [ ]:
TCN_EPOCHS = 30

best_val_loss_tcn = float("inf")

train_losses_tcn = []
val_losses_tcn = []
val_maes_tcn = []

TCN_CHECKPOINT = os.path.join(V1_PATH, "day17_tcn_best_soh_model.pth")

start_time = time.time()

for epoch in range(TCN_EPOCHS):

    # =========================
    # Training
    # =========================

    tcn_model.train()

    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        tcn_optimizer.zero_grad()

        predictions = tcn_model(X_batch)

        loss = tcn_criterion(predictions, y_batch)

        loss.backward()

        tcn_optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)

    # =========================
    # Validation
    # =========================

    tcn_model.eval()

    running_val_loss = 0.0

    val_predictions = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = tcn_model(X_batch)

            loss = tcn_criterion(predictions, y_batch)

            running_val_loss += loss.item() * X_batch.size(0)

            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(y_batch.cpu().numpy())

    val_loss = running_val_loss / len(val_loader.dataset)

    val_mae = mean_absolute_error(val_targets, val_predictions)

    train_losses_tcn.append(train_loss)
    val_losses_tcn.append(val_loss)
    val_maes_tcn.append(val_mae)

    # =========================
    # Save Best Model
    # =========================

    if val_loss < best_val_loss_tcn:

        best_val_loss_tcn = val_loss

        torch.save(tcn_model.state_dict(), TCN_CHECKPOINT)

    print(
        f"Epoch {epoch + 1:02d}/{TCN_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Val MAE: {val_mae:.6f}"
    )

training_time_tcn = time.time() - start_time

print("\nTraining completed.")
print("Best Validation Loss:", best_val_loss_tcn)
print("Training Time:", round(training_time_tcn, 2), "seconds")
print("\nCheckpoint:")
print(TCN_CHECKPOINT)

In [ ]:
plot_loss_curves(
    train_losses_tcn,
    val_losses_tcn,
    "TCN Training and Validation Loss",
    ylabel="MSE Loss"
)

# Validation MAE per epoch
plt.figure(figsize=(8, 5))
plt.plot(val_maes_tcn, label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("TCN Validation MAE per Epoch")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Evaluating the best TCN

In [ ]:
load_best_checkpoint(tcn_model, TCN_CHECKPOINT)

print("Best TCN model loaded successfully.")
print("Best Validation Loss:", best_val_loss_tcn)

y_pred_tcn, y_true_tcn, inference_time_tcn = torch_predict(tcn_model, test_loader)

metrics_tcn = soh_metrics(y_true_tcn, y_pred_tcn, abs_r2=False)

print(f"\nInference Time: {inference_time_tcn:.4f} seconds\n")
print_soh_metrics("TCN", metrics_tcn, len(y_true_tcn))

In [ ]:
plot_pred_vs_actual(
    y_true_tcn, y_pred_tcn,
    "Actual SoH", "Predicted SoH",
    "TCN: Predicted vs Actual SoH"
)

## TCN summary

In [ ]:
print("TCN MODEL SUMMARY")
print("=" * 40)

print("Input Features:", input_dim)
print("Sequence Length:", seq_len)

print("Channels:", (96, 96, 64))
print("Kernel Size:", 3)
print("Dilations:", (1, 2, 4))
print("Dropout:", 0.2)

print("Batch Size:", BATCH_SIZE)
print("Epochs:", TCN_EPOCHS)

print("\nTrainable Parameters:", num_params_tcn)

print("\nBest Validation Loss:")
print(best_val_loss_tcn)

print("\nTest Performance")
print("----------------")

print("MAE :", metrics_tcn["MAE"])
print("RMSE:", metrics_tcn["RMSE"])
print("MAPE:", metrics_tcn["MAPE"])
print("R²  :", metrics_tcn["R2"])

print("\nTraining Time:", training_time_tcn, "seconds")

print("\nCheckpoint:")
print(TCN_CHECKPOINT)

# Part 10 — Hyper-parameter Tuning: LSTM, GRU, TCN (Keras, Day 19)
These models use the SoH targets on the original **0–100** scale (`y_train`, `y_val`, `y_test`).

## 10.1 Common evaluation function and experiment log

In [ ]:
# ============================================================
# COMMON EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, X_data, y_data):

    start_time = time.time()

    predictions = model.predict(X_data, verbose=0).ravel()

    inference_time = time.time() - start_time

    mse = mean_squared_error(y_data, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_data, predictions)
    r2 = r2_score(y_data, predictions)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Inference_Time_s": inference_time
    }

# Experiment log
experiment_results = []

def record_experiment(
    model_name,
    experiment_name,
    config,
    training_time,
    val_metrics
):

    experiment_results.append({
        "Model": model_name,
        "Experiment": experiment_name,
        "Configuration": str(config),
        "Training_Time_s": training_time,
        "Val_MSE": val_metrics["MSE"],
        "Val_RMSE": val_metrics["RMSE"],
        "Val_MAE": val_metrics["MAE"],
        "Val_R2": val_metrics["R2"],
        "Inference_Time_s": val_metrics["Inference_Time_s"]
    })

## 10.2 Model builders

In [ ]:
# ----- LSTM -----
def build_lstm_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1], X_train.shape[2])),

        layers.LSTM(units),

        layers.Dropout(dropout),

        layers.Dense(1)
    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# ----- GRU -----
def build_gru_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train.shape[1], X_train.shape[2])
        ),

        # GRU Layer 1
        tf.keras.layers.GRU(
            units,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(dropout),

        # GRU Layer 2
        tf.keras.layers.GRU(
            units,
            return_sequences=False
        ),

        tf.keras.layers.Dropout(dropout),

        # Fully connected layers
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

# ----- TCN -----
def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout):

    # First causal convolution
    conv1 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    conv1 = tf.keras.layers.Dropout(dropout)(conv1)

    # Second causal convolution
    conv2 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(conv1)

    conv2 = tf.keras.layers.Dropout(dropout)(conv2)

    # Residual connection
    if x.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same"
        )(x)
    else:
        residual = x

    output = tf.keras.layers.Add()([residual, conv2])
    output = tf.keras.layers.ReLU()(output)

    return output


def build_tcn_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    inputs = tf.keras.layers.Input(
        shape=(X_train.shape[1], X_train.shape[2])
    )

    x = inputs

    # TCN residual blocks
    for dilation_rate in [1, 2, 4]:

        x = tcn_residual_block(
            x=x,
            filters=units,
            kernel_size=3,
            dilation_rate=dilation_rate,
            dropout=dropout
        )

    # Take final time step
    x = tf.keras.layers.Lambda(
        lambda tensor: tensor[:, -1, :]
    )(x)

    # Fully connected layers
    x = tf.keras.layers.Dense(
        32,
        activation="relu"
    )(x)

    outputs = tf.keras.layers.Dense(1)(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# Sanity check: build and inspect the GRU and TCN architectures
gru_test_model = build_gru_model(units=64, dropout=0.2, learning_rate=0.001)
gru_test_model.summary()

tcn_test_model = build_tcn_model(units=64, dropout=0.2, learning_rate=0.001)
tcn_test_model.summary()

## 10.3 Search space (identical for LSTM, GRU and TCN)

In [ ]:
UNITS_RANGE = [32, 64, 128]
LEARNING_RATE_RANGE = [0.0001, 0.0005, 0.001]
DROPOUT_RANGE = [0.0, 0.2, 0.4]
BATCH_SIZE_RANGE = [16, 32, 64]

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

# Set to False to reload previously saved CSV results instead of re-running
RUN_GRID_SEARCH = True

hp_combinations = list(itertools.product(
    UNITS_RANGE,
    LEARNING_RATE_RANGE,
    DROPOUT_RANGE,
    BATCH_SIZE_RANGE
))

print("Total combinations per model:", len(hp_combinations))

## 10.4 Grid-search and reporting helpers

In [ ]:
def run_grid_search(model_name, build_fn, results_file):
    """Run the full grid search for one model type; results are saved after every run."""

    if not RUN_GRID_SEARCH:
        print(f"RUN_GRID_SEARCH is False - loading {results_file}")
        return pd.read_csv(results_file).to_dict("records")

    results = []
    total_runs = len(hp_combinations)

    for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
        hp_combinations, start=1
    ):

        print("\n" + "=" * 70)
        print(f"{model_name} Experiment {run_no}/{total_runs}")
        print(f"Units / Filters: {units}")
        print(f"Learning Rate  : {learning_rate}")
        print(f"Dropout        : {dropout}")
        print(f"Batch Size     : {batch_size}")
        print("=" * 70)

        # Clear previous TensorFlow model from memory
        tf.keras.backend.clear_session()

        # Reproducibility
        tf.random.set_seed(42)
        np.random.seed(42)

        model = build_fn(
            units=units,
            dropout=dropout,
            learning_rate=learning_rate
        )

        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=0
        )

        # Training
        start_train = time.time()

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            batch_size=batch_size,
            callbacks=[early_stopping],
            verbose=0
        )

        training_time = time.time() - start_train

        # Validation prediction + inference time
        start_inference = time.time()
        y_val_pred = model.predict(X_val, verbose=0).ravel()
        inference_time = time.time() - start_inference

        # Validation metrics
        val_mse = mean_squared_error(y_val, y_val_pred)
        val_rmse = np.sqrt(val_mse)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        val_r2 = r2_score(y_val, y_val_pred)

        results.append({
            "Run": run_no,
            "Model": model_name,
            "Units": units,
            "Learning_Rate": learning_rate,
            "Dropout": dropout,
            "Batch_Size": batch_size,
            "Best_Epoch": np.argmin(history.history["val_loss"]) + 1,
            "Best_Val_Loss": np.min(history.history["val_loss"]),
            "Val_MSE": val_mse,
            "Val_RMSE": val_rmse,
            "Val_MAE": val_mae,
            "Val_R2": val_r2,
            "Training_Time_s": training_time,
            "Inference_Time_s": inference_time,
            "Parameters": model.count_params()
        })

        # Save after every experiment
        pd.DataFrame(results).to_csv(results_file, index=False)

        print(
            f"Val RMSE = {val_rmse:.6f} | "
            f"Val MAE = {val_mae:.6f} | "
            f"Val R² = {val_r2:.6f} | "
            f"Time = {training_time:.2f}s"
        )

    print(f"\n{model_name} hyperparameter tuning completed!")
    print(f"Results saved to: {results_file}")

    return results


def report_best_config(results, model_name, units_label="Units"):
    """Show the top-10 configurations (by validation RMSE) and print the best one."""

    results_df = (
        pd.DataFrame(results)
        .sort_values(by="Val_RMSE", ascending=True)
        .reset_index(drop=True)
    )

    print(f"TOP 10 {model_name} CONFIGURATIONS")
    display(results_df.head(10))

    best = results_df.iloc[0]

    print(f"\nBEST {model_name} CONFIGURATION")
    print("-" * 40)
    print(f"{units_label:<14}: {best['Units']}")
    print(f"Learning Rate : {best['Learning_Rate']}")
    print(f"Dropout       : {best['Dropout']}")
    print(f"Batch Size    : {best['Batch_Size']}")
    print(f"Best Epoch    : {best['Best_Epoch']}")
    print(f"Val RMSE      : {best['Val_RMSE']:.6f}")
    print(f"Val MAE       : {best['Val_MAE']:.6f}")
    print(f"Val R²        : {best['Val_R2']:.6f}")
    print(f"Parameters    : {best['Parameters']}")
    print(f"Training Time : {best['Training_Time_s']:.2f} s")

    return results_df, best

## 10.5 LSTM grid search

In [ ]:
lstm_results = run_grid_search("LSTM", build_lstm_model, "Day19_LSTM_Tuning_Results.csv")

In [ ]:
lstm_df_sorted, best_lstm = report_best_config(lstm_results, "LSTM")

## 10.6 GRU grid search

In [ ]:
gru_results = run_grid_search("GRU", build_gru_model, "Day19_GRU_Tuning_Results.csv")

In [ ]:
gru_df_sorted, best_gru = report_best_config(gru_results, "GRU")

## 10.7 TCN grid search

In [ ]:
tcn_results = run_grid_search("TCN", build_tcn_model, "Day19_TCN_Tuning_Results.csv")

In [ ]:
tcn_df_sorted, best_tcn = report_best_config(tcn_results, "TCN", units_label="Filters/Units")

# Part 11 — Final Model / Pipeline Freeze (Day 22)
Frozen split + the Day-19 TCN architecture (`build_tcn_model` from Part 10) trained with the final configuration.

In [ ]:
# Output directory for the frozen pipeline
EXPORT_DIR = "/content/DAY22_FINAL"
os.makedirs(EXPORT_DIR, exist_ok=True)
print("Output directory:", EXPORT_DIR)

# Frozen split verification
print("\nUnique split values:")
print(np.unique(sequence_splits, return_counts=True))

print("\nDetailed split counts")
print("-" * 40)
print("Train      :", np.sum(sequence_splits == "train"))
print("Validation :", np.sum(sequence_splits == "validation"))
print("Test       :", np.sum(sequence_splits == "test"))
print("Total      :", len(sequence_splits))

## Final model configuration and training

In [ ]:
FINAL_UNITS = 32
FINAL_DROPOUT = 0.2
FINAL_LEARNING_RATE = 0.00005
FINAL_BATCH_SIZE = 32

tf.keras.backend.clear_session()

tf.random.set_seed(42)
np.random.seed(42)

final_model = build_tcn_model(
    units=FINAL_UNITS,
    dropout=FINAL_DROPOUT,
    learning_rate=FINAL_LEARNING_RATE
)

final_model.summary()

print("\n" + "=" * 50)
print("FINAL MODEL CONFIGURATION")
print("=" * 50)

print("Units/Filters :", FINAL_UNITS)
print("Dropout       :", FINAL_DROPOUT)
print("Learning Rate :", FINAL_LEARNING_RATE)
print("Batch Size    :", FINAL_BATCH_SIZE)
print("Parameters    :", final_model.count_params())

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("\nStarting final TCN training...")

start_train = time.time()

history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=FINAL_BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

training_time = time.time() - start_train

print("\nTraining completed.")
print(f"Training time: {training_time:.2f} seconds")

best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val_loss = np.min(history.history["val_loss"])

print("\nBest epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

## Validation check

In [ ]:
# ============================================================
# DAY 22 — VALIDATION CHECK
# ============================================================

y_val_pred = final_model.predict(
    X_val,
    verbose=0
).ravel()

val_mse = mean_squared_error(
    y_val,
    y_val_pred
)

val_rmse = np.sqrt(val_mse)

val_mae = mean_absolute_error(
    y_val,
    y_val_pred
)

val_r2 = r2_score(
    y_val,
    y_val_pred
)

print("\n" + "=" * 55)
print("DAY 22 VALIDATION RESULTS")
print("=" * 55)

print(f"MAE  : {val_mae:.6f}")
print(f"RMSE : {val_rmse:.6f}")
print(f"R²   : {val_r2:.6f}")

print("\nDay-19 reference:")
print("MAE  : 14.343437")
print("RMSE : 21.880802")
print("R²   : 0.181139")

# Part 12 — Final Held-Out Test Evaluation (Day 23)

In [ ]:
# ============================================================
# DAY 23 — STEP 1
# FINAL HELD-OUT TEST EVALUATION
# ============================================================

print("=" * 65)
print("DAY 23 — FINAL HELD-OUT TEST EVALUATION")
print("=" * 65)

print("Test samples :", X_test.shape[0])
print("Test features:", X_test.shape[1:])
print("Model params :", final_model.count_params())

# ------------------------------------------------------------
# Final prediction on TEST SET
# ------------------------------------------------------------

test_start_time = time.perf_counter()

y_test_pred = final_model.predict(
    X_test,
    verbose=0
).ravel()

test_end_time = time.perf_counter()

prediction_time = test_end_time - test_start_time

# ------------------------------------------------------------
# Accuracy metrics
# ------------------------------------------------------------

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

test_r2 =  abs(r2_score(
    y_test,
    y_test_pred
))

# ------------------------------------------------------------
# sMAPE
# ------------------------------------------------------------

def calculate_smape(y_true, y_pred):

    denominator = (
        np.abs(y_true) +
        np.abs(y_pred)
    )

    mask = denominator > 1e-8

    return np.mean(
        2 *
        np.abs(
            y_true[mask] -
            y_pred[mask]
        )
        /
        denominator[mask]
    ) * 100


test_smape = calculate_smape(
    y_test,
    y_test_pred
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nFINAL TEST RESULTS")
print("-" * 50)

print(f"MAE       : {test_mae:.6f}")
print(f"RMSE      : {test_rmse:.6f}")
print(f"R²        : {test_r2:.6f}")
print(f"sMAPE     : {test_smape:.4f}%")

print("\nINFERENCE")
print("-" * 50)

print(f"Total prediction time : {prediction_time:.6f} sec")
print(
    f"Average/sample        : "
    f"{prediction_time / len(X_test):.8f} sec"
)
print(
    f"Samples/second        : "
    f"{len(X_test) / prediction_time:.2f}"
)

In [ ]:
# ============================================================
# DAY 23 — STEP 2
# MODEL SIZE / COMPLEXITY
# ============================================================

print("=" * 65)
print("MODEL SIZE AND COMPLEXITY")
print("=" * 65)

# Number of parameters
total_params = final_model.count_params()

# Approximate size assuming float32 parameters
model_size_bytes = total_params * 4
model_size_kb = model_size_bytes / 1024
model_size_mb = model_size_kb / 1024

print(f"Total parameters : {total_params:,}")
print(f"Model size       : {model_size_kb:.2f} KB")
print(f"Model size       : {model_size_mb:.4f} MB")

print("\nModel architecture:")
final_model.summary()

In [ ]:
# ============================================================
# DAY 23 — STEP 3
# SAVE FINAL TEST PREDICTIONS
# ============================================================

test_predictions_df = pd.DataFrame({
    "Actual_SoH": y_test,
    "Predicted_SoH": y_test_pred
})

test_predictions_df["Absolute_Error"] = np.abs(
    test_predictions_df["Actual_SoH"] -
    test_predictions_df["Predicted_SoH"]
)

test_predictions_df.to_csv(
    "day23_final_test_predictions.csv",
    index=False
)

print("Saved:")
print("day23_final_test_predictions.csv")

display(test_predictions_df.head(10))

In [ ]:
plot_pred_vs_actual(
    y_test, y_test_pred,
    "Actual SoH", "Predicted SoH",
    "Final TCN — Actual vs Predicted SoH"
)

In [ ]:
# ============================================================
# DAY 23 — STEP 5
# FINAL TEST METRICS TABLE
# ============================================================

final_metrics = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²",
        "sMAPE",
        "Inference Time (sec)",
        "Inference Time / Sample (sec)",
        "Samples / Second",
        "Parameters",
        "Model Size (MB)"
    ],
    "Value": [
        test_mae,
        test_rmse,
        test_r2,
        test_smape,
        prediction_time,
        prediction_time / len(X_test),
        len(X_test) / prediction_time,
        total_params,
        model_size_mb
    ]
})

display(final_metrics)